# Set G — Small Networks: STM & WTA (LEGO–Colab)

**Final (Option 1 style)** — 2026-03-09\
**Author:** Satish Nair

Run **G0 → G5**. Includes systems‑theoretic callouts + Try with Copilot prompts.

---
## Table of Contents
- [G0 Starter](#g0)
- [G1 STM (excitatory)](#g1)
- [G2 STM (inhibitory)](#g2)
- [G3 WTA baseline (no inhibition)](#g3)
- [G4 WTA with global inhibition](#g4)
- [G5 Contrast vs decision time](#g5)
- [Reflection](#reflection)
---

## G0 — Starter (run once) <a id='g0'></a>

In [ ]:

!pip -q install neuron==8.2.4
try: from neuron import h, gui
except: from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt
h.load_file('stdrun.hoc')

s0=h.Section(name='bootstrap')

def plot_traces(t_vec,Vm_mat,labels,title='',figsize=(7,3)):
    plt.figure(figsize=figsize)
    for vm,l in zip(Vm_mat,labels): plt.plot(t_vec,vm,label=l)
    plt.xlabel('Time (ms)'); plt.ylabel('V (mV)'); plt.title(title)
    if len(labels)<=10: plt.legend(ncol=2,fontsize=8)
    plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

def mk_rec(sec): v=h.Vector(); v.record(sec(0.5)._ref_v); return v

print('G0 ready. Proceed to G1.')


### Shared helpers (auto-loaded)

In [ ]:

def build_population(N=5,name_prefix='cell'):
    cells=[]
    for i in range(N):
        s=h.Section(name=f"{name_prefix}{i}")
        s.L=s.diam=20; s.Ra=100; s.cm=1; s.insert('hh'); cells.append(s)
    return cells

def connect_all_to_all(cells,kind='E',tau=2.0,e_rev=0.0,weight=0.005,delay=1.0):
    syns=[]; ncs=[]
    for i,pre in enumerate(cells):
        for j,post in enumerate(cells):
            if i==j: continue
            syn=h.ExpSyn(post(0.5)); syn.tau=tau; syn.e=e_rev
            nc=h.NetCon(pre(0.5)._ref_v,syn,sec=pre)
            nc.threshold=-20; nc.delay=delay; nc.weight[0]=weight
            syns.append(syn); ncs.append(nc)
    return syns,ncs

def brief_iclamp(sec,delay=5,dur=2,amp=0.4): ic=h.IClamp(sec(0.5)); ic.delay=delay; ic.dur=dur; ic.amp=amp; return ic

def tonic_iclamp(sec,amp=0.1,delay=2,dur=1e9): ic=h.IClamp(sec(0.5)); ic.delay=delay; ic.dur=dur; ic.amp=amp; return ic

def run_and_record(cells,tstop=200,vlabels=None):
    t=h.Vector(); t.record(h._ref_t)
    recs=[mk_rec(c) for c in cells]
    h.finitialize(-65*mV); h.continuerun(tstop*ms)
    tnp=np.array(t); vm=[np.array(r) for r in recs]
    if vlabels is None: vlabels=[f"V[{i}]" for i in range(len(cells))]
    plot_traces(tnp,vm,vlabels)
    return tnp,vm

def time_to_win(t,traces):
    t=np.array(t); idx=t>=(t.max()-20)
    means=np.array([np.mean(v[idx]) for v in traces])
    w=int(np.argmax(means))
    V=np.vstack(traces); lead=V[w]-np.max(np.delete(V,w,axis=0),axis=0)
    try: tstar=t[np.where(lead>3)[0][0]]
    except: tstar=np.nan
    return w,tstar,means


## G1 — STM (excitatory all-to-all) <a id='g1'></a>

In [ ]:

N=5
cells_E=build_population(N)
synsE,ncsE=connect_all_to_all(cells_E,kind='E',tau=2.0,e_rev=0.0,weight=0.006,delay=1.0)
drive0=brief_iclamp(cells_E[0],delay=5,dur=2,amp=0.4)
t,vm=run_and_record(cells_E,tstop=200,vlabels=[f'E[{i}]' for i in range(N)])
print('Tip: Increase weight to prolong reverberation (STM).')


> **Systems callout — Reverberation loop**
- Positive feedback via E‑E connections sustains transient activity.
- Stability depends on weight × synaptic decay vs membrane leak.
- Too strong → runaway; too weak → rapid decay.

**Try with Copilot:**
> Predict the critical weight where STM transitions from brief echo to sustained reverberation.

## G2 — STM (inhibitory all-to-all) <a id='g2'></a>

In [ ]:

N=5
cells_I=build_population(N)
synsI,ncsI=connect_all_to_all(cells_I,kind='I',tau=8.0,e_rev=-75.0,weight=0.01,delay=1.0)
drive0I=brief_iclamp(cells_I[0],delay=5,dur=2,amp=0.4)
t,vm=run_and_record(cells_I,tstop=200,vlabels=[f'I[{i}]' for i in range(N)])
print('Increase inhibitory weight to quench activity faster.')


> **Systems callout — Damped dynamics**
- I‑I coupling pushes voltages toward E_I, shortening activity tails.
- Illustrates how inhibition stabilizes or silences networks.

**Try with Copilot:**
> Estimate the inhibitory weight needed to fully suppress post-pulse activity within 20 ms.

## G3 — WTA baseline (excitatory only) <a id='g3'></a>

In [ ]:

N=5
cells_WE=build_population(N)
synsWE,ncsWE=connect_all_to_all(cells_WE,kind='E',tau=2.0,e_rev=0.0,weight=0.003,delay=1.0)
amps=np.linspace(0.08,0.12,N)
ics=[tonic_iclamp(cells_WE[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
t,vm=run_and_record(cells_WE,tstop=400,vlabels=[f'Eonly[{i}]' for i in range(N)])
w,tstar,means=time_to_win(t,vm)
print(f'Baseline: winner={w}, time_to_win≈{tstar}, means={means}')


> **Systems callout — No inhibition → weak competition**
- Pure E‑E coupling lacks mechanism to suppress non‑winners.
- All units remain active; small differences lead to soft biases.

**Try with Copilot:**
> Explain why E‑only networks rarely produce sharp WTA without nonlinearities or adaptation.

## G4 — WTA with global inhibition <a id='g4'></a>

In [ ]:

N=5
princ=build_population(N)
inh=h.Section(name='inh'); inh.L=inh.diam=18; inh.Ra=100; inh.cm=1; inh.insert('hh')
amps=np.linspace(0.08,0.12,N)
ics=[tonic_iclamp(princ[i],amp=float(amps[i]),delay=5.0) for i in range(N)]

# E→I
e2i_syn=[]; e2i_nc=[]
for p in princ:
    syn=h.ExpSyn(inh(0.5)); syn.tau=2.0; syn.e=0.0
    nc=h.NetCon(p(0.5)._ref_v,syn,sec=p); nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.006
    e2i_syn.append(syn); e2i_nc.append(nc)
# I→E
i2e_syn=[]; i2e_nc=[]
for p in princ:
    syn=h.ExpSyn(p(0.5)); syn.tau=8.0; syn.e=-75.0
    nc=h.NetCon(inh(0.5)._ref_v,syn,sec=inh); nc.threshold=-20; nc.delay=0.8; nc.weight[0]=0.02
    i2e_syn.append(syn); i2e_nc.append(nc)

_synSE,_ncSE=connect_all_to_all(princ,kind='E',tau=2.0,e_rev=0.0,weight=0.0015,delay=1.0)

t=h.Vector(); t.record(h._ref_t)
recP=[mk_rec(p) for p in princ]; recI=mk_rec(inh)

h.finitialize(-65*mV); h.continuerun(500*ms)
tnp=np.array(t); vP=[np.array(r) for r in recP]; vI=np.array(recI)
plot_traces(tnp,vP,[f'P[{i}]' for i in range(N)],title='G4: WTA with global inhibition')
w,tstar,means=time_to_win(tnp,vP)
print(f'Winner={w}, time_to_win≈{tstar}, means={means}')


> **Systems callout — Competition via inhibition**
- E→I→E loop creates global suppression.
- Small input advantages get amplified as inhibition silences non‑winners.

**Try with Copilot:**
> Increase global inhibitory weight: predict effect on decision speed and winner identity.

## G5 — Contrast vs decision time <a id='g5'></a>

In [ ]:

def wta_decision_time(contrast=0.04,base=0.08,N=5,inh_w=0.02):
    princ=build_population(N); inh=h.Section(name='inh2'); inh.L=inh.diam=18; inh.Ra=100; inh.cm=1; inh.insert('hh')
    amps=np.linspace(base,base+contrast,N)
    ics=[tonic_iclamp(princ[i],amp=float(amps[i]),delay=5.0) for i in range(N)]
    for p in princ:
        se=h.ExpSyn(inh(0.5)); se.tau=2.0; se.e=0.0
        nc=h.NetCon(p(0.5)._ref_v,se,sec=p); nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.006
    for p in princ:
        si=h.ExpSyn(p(0.5)); si.tau=8.0; si.e=-75.0
        nc=h.NetCon(inh(0.5)._ref_v,si,sec=inh); nc.threshold=-20; nc.delay=0.8; nc.weight[0]=inh_w
    t=h.Vector(); t.record(h._ref_t)
    recP=[mk_rec(p) for p in princ]
    h.finitialize(-65*mV); h.continuerun(500*ms)
    tnp=np.array(t); vP=[np.array(r) for r in recP]
    return time_to_win(tnp,vP)

contrasts=[0.005,0.01,0.02,0.03,0.04]
times=[]
for c in contrasts:
    w,tstar,_=wta_decision_time(contrast=c)
    times.append(tstar)
plt.figure(figsize=(4.8,3))
plt.plot(contrasts,times,'o-')
plt.xlabel('Input contrast (Δamp)'); plt.ylabel('Time-to-win (ms)'); plt.title('G5: WTA decision time vs contrast')
plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()


> **Systems callout — Contrast → speed**
- Larger contrast accelerates divergence between competitors.
- Global inhibition converts small drive differences into fast decisions.

**Try with Copilot:**
> Sweep contrasts more finely and fit an approximate inverse relationship: time-to-win ∝ 1/contrast.

## Reflection <a id='reflection'></a>
- Why does E‑E feedback support STM? How does leak counteract it?
- Which parameters most strongly shape WTA speed: E→I, I→E, Δinput, or τ?
- How could we add realistic noise and test robustness of decisions?